In [45]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Optional,Tuple,Dict
from __future__ import annotations

In [46]:
def find_tesis_root(start:Path |None = None, marker:str = "Results")-> Path:
    p = (start or Path.cwd()).resolve()
    for parent in [p, *p.parents]:
        if (parent / "Results").exists():
            return parent
    raise FileNotFoundError("Results forlder not found")

TESIS_ROOT = find_tesis_root()

In [ ]:
Results_Root = TESIS_ROOT / "Results"
colmap = {
    "t": "Time in s",
    "f": "Electrical Frequency in Hz",
    "V": "Voltage, Magnitude in p.u.",
    "w_eqsg": "Speed in p.u.",
    "P":"Active Power in MW",
    "Q":"Reactive Power in Mvar",
    "Ipos": "Positive-Sequence Current, Magnitude in p.u.",
}




In [48]:
# HELPERS

def load_pf_csv(path: str,colmap) -> pd.DataFrame:
    """
    Load a CSV exported by PowerFactory
    """
    df = pd.read_csv(path)
    df.columns = df.iloc[0]
    df = df.iloc[1:].reset_index(drop = True)
    #colmap = resolve_colmap(df,colmap)
    df = prepare_df(df,colmap)
    return df

def prepare_df(df:pd.DataFrame,colmap:dict)-> pd.DataFrame:
    for k,col in colmap.items():
        if col in df.columns:
            df[col] = pd.to_numeric(df[col],errors = "coerce")
    return df

def get_series(df: pd.DataFrame, colmap:Dict[str,str], key:str)-> pd.Series:
    """
    Receives a Dataframe in order to search for the column
    Colmap is a dictionary, where the mapping of the desired variable with the 
        respective column will happen i.e. in Dict {f:Electrical frequency}
        I search for the key "f"
    key: name of the variable i want to search for
    """
    if key not in colmap:
        raise KeyError(f"Missing key: {key} in colmap")
    col = colmap[key]
    if col not in df.columns:
        print(f"Column {col} not found in the CSV")
        print(f"Available columns: {df.columns}")
    return df[col]
    

def window_mask(t:np.ndarray,t0:float,t1:float) -> np.ndarray:
    return(t >= t0) & (t<=t1)

def finite_diff_derivative(t: np.ndarray, y:np.ndarray) ->np.ndarray:

    """
    This function calculates the derivative dy/dt
    Return an array of the same length of y
    """
    dydt = np.gradient(y,t)
    return dydt

def plot_signal_with(t,y,reference = 50,nadir = None,band = None,settling_time = None, title = "Signal",y_label = "Value"):
    plt.figure(figsize=(8,4))
    plt.plot(t,y,label = "signal")

    if nadir is not None:
        plt.axhline(nadir,linestyle="--",label = f"freq. nadir: {nadir} Hz",linewidth=0.7)
    
    if reference is not None:
        plt.axhline(reference + band, linestyle=":", color="gray",linewidth = 0.7)
        plt.axhline(reference - band, linestyle=":", color="gray",linewidth = 0.7)

    plt.axvline(settling_time, linestyle="--", linewidth=0.7)
    plt.xlabel("Time [s]")
    plt.ylabel(y_label)
    plt.title(title)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

def plot_signal(x,y,title:str = "Signal",y_label:str = "y label",label:str ="label"):
    plt.figure(figsize=(8,4))
    plt.plot(x,y,label = label)
    plt.xlabel("Time [s]")
    plt.ylabel(y_label)
    plt.title(title)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

def load_all_results(results_root: str | Path,
                      colmap) -> Dict[Tuple[str, str], pd.DataFrame]:
    """
    Walks a Results folder with structure:
      Results/
        <TECH>/
          <CASE>/
            <CASE>.csv 

    Returns:
      dict keyed by (tech_name, case_name) -> DataFrame
    """
    results_root = Path(results_root)
    if not results_root.exists():
        raise FileNotFoundError(f"Results root not found: {results_root}")

    out: Dict[Tuple[str, str], pd.DataFrame] = {}

    tech_dirs = [p for p in results_root.iterdir() if p.is_dir()]
    for tech_dir in tech_dirs:
        if not tech_dir.is_dir():
            continue
        case_dirs = tech_dir.iterdir()
        for case_dir in case_dirs:
            if not case_dir.is_dir():
                continue
            tech_name = tech_dir.name
            case_name = case_dir.name
            for csv_path in case_dir.glob("*.csv"):
                df = load_pf_csv(csv_path,colmap)
                out[(tech_name,case_name)] = df
    return out


In [49]:
#LOAD STEP
def kpi_f_nadir(df, colmap, t0: float, t1: float ) -> float:
    t= get_series(df,colmap,"t").to_numpy()
    f = get_series(df,colmap,"f").to_numpy()
    m = window_mask(t, t0, t1)
    return float(f[m].min())
 
def kpi_rocof_max(df,colmap,t0:float,t1:float) -> float:
    t=get_series(df,colmap,"t").to_numpy()
    f=get_series(df,colmap,"f").to_numpy()
    m = window_mask(t,t0,t1)
    dfdt = finite_diff_derivative(t[m],f[m]) #df/dt
    idx = np.argmax(np.abs(dfdt))
    rocof = float(dfdt[idx]) 
    return rocof,dfdt,t[m]

def kpi_settling_time_f(df,colmap,t_event:float,
                        f_ref:float =50.0, band_hz:float = 0.5,
                        hold_s:float = 0.5,t_end:float = 20):
    t = get_series(df,colmap,"t").to_numpy()
    f = get_series(df, colmap,"f").to_numpy()
    m = window_mask(t,t_event,t_end)

    tt, ff = t[m],f[m]
    
    inside = np.abs(ff - f_ref) <= band_hz

    for i in range(len(tt)):
        if not inside[i]:
            continue
        t_i = tt[i]
        m_hold = (tt >= t_i) & (tt <= t_i + hold_s)
        if m_hold.any() and inside[m_hold].all():
            return float(t_i - t_event)

    return None  # no asienta

def kpi_f_overshoot(df, colmap, t_event: float, T: float = 10.0) -> float:
    t = get_series(df, colmap, "t").to_numpy()
    f = get_series(df, colmap, "f").to_numpy()
    m = window_mask(t, t_event, t_event + T)
    return float(f[m].max())


def kpi_delta_P(df, colmap, t_event: float,
                pre=( -1.0, -0.1), post=( 1.0, 3.0)) -> float:
    t = get_series(df, colmap, "t").to_numpy()
    P = get_series(df, colmap, "P").to_numpy()

    mpre  = window_mask(t, t_event + pre[0],  t_event + pre[1])
    mpost = window_mask(t, t_event + post[0], t_event + post[1])

    Ppre  = P[mpre].mean()
    Ppost = P[mpost].mean()
    return float(Ppost - Ppre)

def kpi_settling_time_P(df, colmap, t_event: float,
                        band_frac: float = 0.01, hold_s: float = 0.5,
                        post=(1.0, 3.0), t_end: float = 20.0):
    t = get_series(df, colmap, "t").to_numpy()
    P = get_series(df, colmap, "P").to_numpy()

    # steady-state target from post-window average
    mpost = window_mask(t, t_event + post[0], t_event + post[1])
    P_ref = float(P[mpost].mean())
    band = band_frac * abs(P_ref)

    m = window_mask(t, t_event, t_end)
    tt, PP = t[m], P[m]
    inside = np.abs(PP - P_ref) <= band

    for i in range(len(tt)):
        if not inside[i]:
            continue
        t_i = tt[i]
        m_hold = (tt >= t_i) & (tt <= t_i + hold_s)
        if m_hold.any() and inside[m_hold].all():
            return float(t_i - t_event)
    return None

def kpi_eqsg_speed_minmax(df, colmap, t0: float, t1: float):
    t = get_series(df, colmap, "t").to_numpy()
    w = get_series(df, colmap, "w_eqsg").to_numpy()
    m = window_mask(t, t0, t1)
    return float(w[m].min()), float(w[m].max())

In [50]:
#3PHSC
def kpi_Vmin_fault(df, colmap, t_fault: float, t_clear: float) -> float:
    t = get_series(df, colmap, "t").to_numpy()
    V = get_series(df, colmap, "V").to_numpy()
    m = window_mask(t, t_fault, t_clear)
    return float(V[m].min())

def kpi_V_recovery_time(df, colmap, t_clear: float, Vthr: float = 0.95,
                        exclude_s: float = 0.05, hold_s: float = 0.2,
                        t_end: float = 20.0):
    t = get_series(df, colmap, "t").to_numpy()
    V = get_series(df, colmap, "V").to_numpy()

    m = window_mask(t, t_clear + exclude_s, t_end)
    tt, VV = t[m], V[m]
    ok = VV >= Vthr

    for i in range(len(tt)):
        if not ok[i]:
            continue
        t_i = tt[i]
        m_hold = (tt >= t_i) & (tt <= t_i + hold_s)
        if m_hold.any() and ok[m_hold].all():
            return float(t_i - t_clear)
    return None

def kpi_Ipos_max(df, colmap, t0: float, t1: float) -> float:
    t = get_series(df, colmap, "t").to_numpy()
    I = get_series(df, colmap, "Ipos").to_numpy()
    m = window_mask(t, t0, t1)
    return float(I[m].max())

def kpi_Pmin_fault(df, colmap, t0: float, t1: float) -> float:
    t = get_series(df, colmap, "t").to_numpy()
    P = get_series(df, colmap, "P").to_numpy()
    m = window_mask(t, t0, t1)
    return float(P[m].min())
def kpi_P_recovery_time(df, colmap, t_clear: float,
                        alpha: float = 0.95,
                        pre=(-1.0, -0.1),
                        exclude_s: float = 0.05,
                        hold_s: float = 0.2,
                        t_end: float = 20.0):
    t = get_series(df, colmap, "t").to_numpy()
    P = get_series(df, colmap, "P").to_numpy()

    # pre-fault reference
    mpre = window_mask(t, t_clear + pre[0], t_clear + pre[1])
    Ppre = float(P[mpre].mean())
    target = alpha * Ppre

    m = window_mask(t, t_clear + exclude_s, t_end)
    tt, PP = t[m], P[m]
    ok = PP >= target

    for i in range(len(tt)):
        if not ok[i]:
            continue
        t_i = tt[i]
        m_hold = (tt >= t_i) & (tt <= t_i + hold_s)
        if m_hold.any() and ok[m_hold].all():
            return float(t_i - t_clear)
    return None



In [51]:

out = load_all_results(Results_Root,colmap)
loadsteps = {}
three_ph_scs = {}
base_cases = {}
for (tech,name),df in out.items():
    if "Load_Step" in name:
        loadsteps[(tech,name)] = df
    elif "3PhSC" in name: 
        three_ph_scs[(tech,name)] = df
    else:
        base_cases[(tech,name)] = df

print(loadsteps.keys())
print(three_ph_scs.keys())

TypeError: unhashable type: 'list'

In [ ]:
for (technology,fault),df in loadsteps.items():
    Q = get_series(df,colmap,"P")
    t = get_series(df,colmap,"t")
    RoCoF,dfdt,t_rocof = kpi_rocof_max(df,colmap,0,20)
    plot_signal(t,Q,"Active Power","MW",technology)
    #plot_signal(t_rocof,dfdt,"RoCoF","Hz/s",technology)
    

In [ ]:
rows = []
for (technology,fault), df in loadsteps.items():
    nadir = kpi_f_nadir(df,colmap,0.95,15)
    RoCoF,dfdt,t_rocof = kpi_rocof_max(df,colmap,1.020,3)
    settling_time = kpi_settling_time_f(df,colmap,1,50,0.5,0.5,20)
    row = {"Technology" : technology, "f nadir [Hz]": nadir,
           "Rocof [Hz/s]":RoCoF,"f settling time [s]": settling_time}
    rows.append(row)
    t = get_series(df,colmap,"t")
    f = get_series(df,colmap,"f")
    #plot_signal(t,f,50,nadir,0.5,settling_time,"Frequency","Hz")
    #plot_signal(t_rocof,dfdt,reference = 0.0,nadir = None,band = 0,settling_time=0,title="RoCoF",y_label="Hz/s")

loadstep_analysis = pd.DataFrame(rows)
loadstep_analysis.head()

In [ ]:
rows = []
for (technology,fault), df in three_ph_scs.items():
    Vdip = kpi_Vmin_fault(df,colmap,0.95,20)
    recovery_time = kpi_V_recovery_time(df,colmap,1.1,0.95,0.05,1,20)
    Ipos_max = kpi_Ipos_max(df,colmap,0,20)
    nadir = kpi_f_nadir(df,colmap,0,20)
    RoCoF,dfdt,t_rocof = kpi_rocof_max(df,colmap,0,5)


    row = {"Technology" : technology, "Voltage dip [V p.u.]": Vdip,
           "Recovery time [s]":recovery_time,"Max Positive Current": Ipos_max,
           "Frequency nadir [Hz]":nadir,"RoCoF [Hz/s]":RoCoF}
    rows.append(row)
    t = get_series(df,colmap,"t")
    V = get_series(df,colmap,"V")
    f= get_series(df,colmap,"f")
    #plot_signal(t,V,reference = 1,nadir = Vdip,band = 0.5,settling_time=recovery_time,
                #title="Voltage",y_label="Voltage [p.u.]")
    #plot_signal(t_rocof,dfdt,0,0,0,0,"RoCoF","Hz/s")

three_ph_sc_analysis = pd.DataFrame(rows)
three_ph_sc_analysis.head(7)


In [ ]:
print(
    loadstep_analysis.to_string(
        index=False,
        justify="center",
        float_format="{:.3f}".format
    )
)


In [ ]:
print(
    three_ph_sc_analysis.to_string(
        index=False,
        justify="center",
        float_format="{:.3f}".format
    )
)
